# 05 — Low-Rank RNN with Flax NNX

A **Low-Rank RNN** constrains the recurrent connectivity matrix to have rank r:

$$J = M N^\top, \quad M, N \in \mathbb{R}^{N \times r}, \quad r \ll N$$

with element-wise form $J_{ij} = \sum_{r=1}^{R} m_i^{(r)} n_j^{(r)}$.

**Why?**
- Network dynamics are confined to an r-dimensional subspace — interpretable
- Strong regularisation: fewer parameters than full J
- Motivated by neuroscience: cortical activity is low-dimensional
- Analytically tractable: mean-field theory applies

In flax.nnx, M and N are stored as `nnx.Param` attributes, and the recurrent computation `h @ N @ M.T` is differentiable and scannable.

In [ ]:
import sys; sys.path.insert(0, '..')
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt

from src.rnn_jax import LowRankRNNCell, LowRankRNN, LowRankRNNModel, VanillaRNNModel
from src.utils import make_sinusoid_task, plot_loss_curves, plot_eigenspectrum
from src import train as T

## 5.1 Low-Rank Parameterisation

Unlike a vanilla RNN where `J ∈ R^{N×N}` has N² parameters, the low-rank RNN stores only `M` and `N` — each of size N×r.

In [ ]:
N, r = 64, 4
cell = LowRankRNNCell(input_size=1, hidden_size=N, rank=r, rngs=nnx.Rngs(0))

print('M shape:', cell.M.shape)   # (N, r)
print('N shape:', cell.N.shape)   # (N, r)

J_rec = cell.J_rec
print('J_rec shape:', J_rec.shape)      # (N, N)

# Verify rank constraint
singular_values = jnp.linalg.svd(J_rec, compute_uv=False)
print(f'\nSingular values (first 8): {np.array(singular_values[:8]).round(4)}')
print(f'Numerical rank: {int(jnp.sum(singular_values > 1e-6))}  (should be {r})')

# Parameter count comparison
params_full  = N * N              # full J
params_lr    = 2 * N * r          # M + N
print(f'\nFull J: {params_full:,} params')
print(f'Low-rank r={r}: {params_lr:,} params ({100*params_lr/params_full:.1f}% of full)')

In [ ]:
# Eigenspectrum: low-rank structure is visible — most eigenvalues are zero
fig = plot_eigenspectrum(np.array(J_rec), f'Low-rank J_rec (rank={r}, N={N})')
plt.show()

## 5.2 Forward Pass via `lax.scan`

In [ ]:
model_lr = LowRankRNNModel(
    input_size=1, hidden_size=64, rank=4, output_size=1, rngs=nnx.Rngs(0)
)

B, T_len = 8, 50
xs = jnp.ones((B, T_len, 1))   # batch-first: (B, T, I)
preds = model_lr(xs)
print('Output shape:', preds.shape)   # (B, T, 1)

## 5.3 Training: Sinusoid Prediction

Compare low-rank vs vanilla RNN on the same sinusoid prediction task.

In [ ]:
N_SEQ, SEQ_LEN = 512, 100
data = make_sinusoid_task(N_SEQ, SEQ_LEN, key=jax.random.PRNGKey(0))

def loss_fn(model, batch):
    return T.mse_loss(model(batch['inputs']), batch['targets'])

# Low-rank RNN (r=4)
model_lr = LowRankRNNModel(1, 64, rank=4, output_size=1, rngs=nnx.Rngs(0))
print('--- Low-Rank RNN (r=4) ---')
losses_lr = T.fit(model_lr, data, loss_fn=loss_fn, n_steps=500, lr=3e-3, batch_size=64, log_every=100)

# Vanilla RNN (same hidden size)
model_van = VanillaRNNModel(1, 64, output_size=1, rngs=nnx.Rngs(0))
print('--- Vanilla RNN ---')
losses_van = T.fit(model_van, data, loss_fn=loss_fn, n_steps=500, lr=3e-3, batch_size=64, log_every=100)

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(losses_lr, label='Low-rank RNN (r=4)')
plt.semilogy(losses_van, label='Vanilla RNN')
plt.xlabel('Step'); plt.ylabel('MSE loss')
plt.title('Low-rank vs Vanilla RNN on sinusoid prediction')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5.4 `vmap` over Rank: Train Multiple Models Simultaneously

Since NNX state is a pytree, we can use `vmap` to train many low-rank models with different ranks in a single vectorised pass.

In [ ]:
# Create models with different ranks (r = 1, 2, 4, 8)
ranks = [1, 2, 4, 8]
final_losses = {}

for r in ranks:
    m = LowRankRNNModel(1, 32, rank=r, output_size=1, rngs=nnx.Rngs(0))
    ls = T.fit(m, data, loss_fn=loss_fn, n_steps=300, lr=3e-3, batch_size=64, log_every=1000)
    final_losses[r] = ls[-1]
    print(f'rank={r:2d}  final loss={ls[-1]:.5f}')

plt.figure(figsize=(6, 4))
plt.bar([str(r) for r in ranks], [final_losses[r] for r in ranks])
plt.xlabel('Rank r'); plt.ylabel('Final MSE loss')
plt.title('Final loss vs low-rank constraint')
plt.grid(axis='y', alpha=0.3); plt.show()

## 5.5 Rank Constraint Holds Throughout Training

In [ ]:
# After training, verify the rank constraint is maintained by construction
J_rec_trained = model_lr.rnn.J_rec
svs = np.linalg.svd(np.array(J_rec_trained), compute_uv=False)

plt.figure(figsize=(8, 3.5))
plt.semilogy(svs, 'o-', ms=4)
plt.axvline(model_lr.rnn.rank - 0.5, color='red', linestyle='--', label=f'rank={model_lr.rnn.rank}')
plt.xlabel('Singular value index'); plt.ylabel('Singular value')
plt.title('Singular values of J_rec after training — rank constraint holds')
plt.legend(); plt.grid(alpha=0.3); plt.show()